# 02 — Gold Star

Dims, facts, contracts enforced, no orphans. Iceberg-everywhere on CE.

In [1]:
import pathlib
# serving Gold counts
import sys; sys.path.append(str(pathlib.Path(".").resolve().parent) if pathlib.Path(".").name=="notebooks" else ".")
from dotenv import load_dotenv; load_dotenv(dotenv_path=pathlib.Path(".env") if pathlib.Path(".env").exists() else pathlib.Path("../.env"))
import os
from sqlalchemy import create_engine, text
host=os.getenv("POSTGRES_HOST","localhost"); port=os.getenv("POSTGRES_PORT","5433")
user=os.getenv("POSTGRES_STREAMLIT_READER_USER","streamlit_reader"); pw=os.getenv("POSTGRES_STREAMLIT_READER_PASSWORD")
db=os.getenv("POSTGRES_WAREHOUSE_DB","banking_dw")
eng=create_engine(f"postgresql+psycopg2://{user}:{pw}@{host}:{port}/{db}")
with eng.connect() as c:
    for t in ["serving.dim_customer","serving.dim_account","serving.fct_transactions"]:
        print(t, c.execute(text(f"SELECT count(*) FROM {t}")).scalar())


serving.dim_customer 6
serving.dim_account 9
serving.fct_transactions 20


In [2]:
# no orphan facts
import os; from sqlalchemy import create_engine, text
host=os.getenv("POSTGRES_HOST","localhost"); port=os.getenv("POSTGRES_PORT","5433")
user=os.getenv("POSTGRES_STREAMLIT_READER_USER","streamlit_reader"); pw=os.getenv("POSTGRES_STREAMLIT_READER_PASSWORD")
db=os.getenv("POSTGRES_WAREHOUSE_DB","banking_dw")
eng=create_engine(f"postgresql+psycopg2://{user}:{pw}@{host}:{port}/{db}")
with eng.connect() as c:
    print(c.execute(text("SELECT count(*) FROM serving.fct_transactions f LEFT JOIN serving.dim_account d ON f.account_sk=d.account_sk WHERE d.account_sk IS NULL")).scalar())
# must be 0


0


In [3]:
# gold SQL
import pathlib
for f in sorted(pathlib.Path("dbt/banking_dbt/models/gold") if pathlib.Path("dbt/banking_dbt/models/gold").exists() else pathlib.Path("../dbt/banking_dbt/models/gold").glob("*.sql")):
    print(f.name)
    print(pathlib.Path(f).read_text()[:350])


dim_account.sql
{{ config(materialized='incremental', unique_key='account_sk', contract={'enforced': true}) }}
-- dim_account SCD2
with dedup as (
  select account_id, customer_id, branch_id, account_type, status, max(silver_loaded_at) as valid_from from {{ ref('silver_accounts') }} group by 1,2,3,4,5
)
select {{ dbt_utils.generate_surrogate_key(['account_id', 'va
dim_branch.sql
{{ config(materialized='table', contract={'enforced': true}) }}
-- dim_branch Type1 (150 rows)
select cast(branch_id as int) as branch_id, branch_name, city, state, ifsc_code from {{ ref('silver_branches') }}
union all select -1, 'Unknown', 'Unknown', 'Unknown', 'Unknown'

dim_customer.sql
{{ config(
    materialized='incremental',
    unique_key='customer_sk',
    contract={'enforced': true}
) }}
-- Gold dim_customer SCD2 with surrogate key + unknown member -1 (Architecture 7.4)
-- Surrogate: {{ dbt_utils.generate_surrogate_key(['customer_id', 'valid_from']) }} pattern
with dedup as (
  select customer_id, nam

In [4]:
# SCD2 check
import os; from sqlalchemy import create_engine, text
eng=create_engine(f"postgresql+psycopg2://{user}:{pw}@{host}:{port}/{db}")
with eng.connect() as c:
    print(list(c.execute(text("SELECT is_current, COUNT(*) FROM serving.dim_customer GROUP BY 1")).fetchall()))
    print(list(c.execute(text("SELECT is_current, COUNT(*) FROM serving.dim_account GROUP BY 1")).fetchall()))
# unknown -1 member exists


[(False, 1), (True, 5)]
[(False, 1), (True, 8)]
